### 1. Montar Google Drive
Primeiro, precisamos garantir que o Colab tenha acesso à sua pasta no Drive.

In [13]:
from google.colab import drive
import xarray as xr
import os

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 2. Carregar os Arquivos NetCDF
Agora, vamos carregar os arquivos especificados utilizando o `xarray`.

In [14]:
path1 = '/content/drive/MyDrive/- IRB/Vendaval-Base/ERA5_INMET_Paired_ROI_4_Stations.nc'
path2 = '/content/drive/MyDrive/- IRB/Vendaval-Base/v2_inicial.nc'

# Carregando o primeiro dataset
ds_stations = xr.open_dataset(path1)
print("--- Dataset: ERA5_INMET_Paired_ROI_4_Stations ---")
display(ds_stations)

# Carregando o segundo dataset
ds_inicial = xr.open_dataset(path2)
print("\n--- Dataset: v2_inicial ---")
display(ds_inicial)

--- Dataset: ERA5_INMET_Paired_ROI_4_Stations ---


<xarray.Dataset> Size: 7MB
Dimensions:              (station: 4, time: 9497)
Coordinates:
  * station              (station) <U4 64B 'A805' 'A853' 'A854' 'A856'
  * time                 (time) datetime64[ns] 76kB 2000-01-01 ... 2025-12-31
    latitude             (station) float64 32B ...
    longitude            (station) float64 32B ...
    station_lat          (station) float64 32B ...
    station_lon          (station) float64 32B ...
Data variables: (12/23)
    ws_mean              (station, time) float64 304kB ...
    ws_max               (station, time) float64 304kB ...
    ws_std               (station, time) float64 304kB ...
    ws_p90               (station, time) float64 304kB ...
    sin_dir_mean         (station, time) float64 304kB ...
    cos_dir_mean         (station, time) float64 304kB ...
    ...                   ...
    tp_max               (station, time) float64 304kB ...
    d_msl                (station, time) float64 304kB ...
    tp_lag1              (station, time) float64 304kB ...
    tp_lag2              (station, time) float64 304kB ...
    tp_roll3             (station, time) float64 304kB ...
    rajada_max_final_ms  (station, time) float64 304kB ...


--- Dataset: v2_inicial ---


<xarray.Dataset> Size: 13GB
Dimensions:               (valid_time: 9413, latitude: 393, longitude: 391)
Coordinates:
  * valid_time            (valid_time) datetime64[ns] 75kB 2000-01-01 ... 202...
  * latitude              (latitude) float32 2kB -33.85 -33.75 ... 5.25 5.35
  * longitude             (longitude) float32 2kB -73.85 -73.75 ... -34.85
    number                int64 8B ...
Data variables:
    categoria             (valid_time, latitude, longitude) int8 1GB ...
    rajada_max_corrigida  (valid_time, latitude, longitude) float32 6GB ...
    direcao_vento         (valid_time, latitude, longitude) float32 6GB ...

### 3. Extração de Dados por Coordenadas e Tempo
Vamos iterar sobre as estações para extrair os dados correspondentes do dataset em grade (`ds_inicial`).

In [15]:
import pandas as pd

# Lista para armazenar os dados extraídos de cada estação
extracted_data_list = []

for station_id in ds_stations.station.values:
    # 1. Pegar coordenadas da estação específica
    # Usamos station_lat e station_lon do ds_stations
    st_lat = ds_stations.sel(station=station_id).station_lat.values.item()
    st_lon = ds_stations.sel(station=station_id).station_lon.values.item()

    # 2. Extrair do dataset de grade (ds_inicial) usando as coordenadas e o tempo das estações
    # Usamos method='nearest' para encontrar o pixel mais próximo
    ds_grid_subset = ds_inicial.sel(
        latitude=st_lat,
        longitude=st_lon,
        valid_time=ds_stations.time,
        method='nearest'
    )

    # 3. Converter para DataFrame e adicionar o ID da estação para referência
    df_subset = ds_grid_subset.to_dataframe().reset_index()
    df_subset['station'] = station_id

    extracted_data_list.append(df_subset)

# Concatenar todos os resultados em um único DataFrame
df_v2_at_stations = pd.concat(extracted_data_list, ignore_index=True)

print("Dados extraídos do v2_inicial para as coordenadas das estações:")
display(df_v2_at_stations.head())

Dados extraídos do v2_inicial para as coordenadas das estações:


,time,categoria,rajada_max_corrigida,direcao_vento,number,valid_time,latitude,longitude,station
0,2000-01-01,0,14.002425,180.0,0,2000-01-01,-27.85,-53.75,A805
1,2000-01-02,0,12.102600,180.0,0,2000-01-02,-27.85,-53.75,A805
2,2000-01-03,0,10.181101,180.0,0,2000-01-03,-27.85,-53.75,A805
3,2000-01-04,0,8.154307,180.0,0,2000-01-04,-27.85,-53.75,A805
4,2000-01-05,0,8.270441,180.0,0,2000-01-05,-27.85,-53.75,A805


### 4. Merge Final
Agora vamos unir os dados originais das estações com os dados que extraímos da grade.

In [16]:
# 1. Garantir que temos o df_stations_orig limpo
df_stations_orig = ds_stations.to_dataframe().reset_index()

# 2. Re-processar df_v2_at_stations para garantir consistência
extracted_list = []
for station_id in ds_stations.station.values:
    st_lat = ds_stations.sel(station=station_id).station_lat.values.item()
    st_lon = ds_stations.sel(station=station_id).station_lon.values.item()

    ds_grid_subset = ds_inicial.sel(
        latitude=st_lat,
        longitude=st_lon,
        valid_time=ds_stations.time,
        method='nearest'
    )

    df_tmp = ds_grid_subset.to_dataframe().reset_index()
    df_tmp['station'] = station_id

    # Renomear valid_time para time se necessário, removendo duplicatas de 'time' se o xarray as criar
    if 'valid_time' in df_tmp.columns:
        if 'time' in df_tmp.columns:
            df_tmp = df_tmp.drop(columns=['time'])
        df_tmp = df_tmp.rename(columns={'valid_time': 'time'})

    extracted_list.append(df_tmp)

df_v2_clean = pd.concat(extracted_list, ignore_index=True)

# 3. Merge Final
df_final = pd.merge(df_stations_orig, df_v2_clean, on=['time', 'station'], how='inner')

print("Dataset final combinado (Estações + Grade):")
display(df_final.head())
print(f"\nFormato final do DataFrame: {df_final.shape}")

Dataset final combinado (Estações + Grade):


,station,time,ws_mean,ws_max,ws_std,ws_p90,sin_dir_mean,cos_dir_mean,msl_mean,msl_min,...,latitude_x,longitude_x,station_lat,station_lon,categoria,rajada_max_corrigida,direcao_vento,number,latitude_y,longitude_y
0,A805,2000-01-01,6.296078,7.452370,0.593588,7.233298,0.739596,-0.659417,100868.335938,100646.3750,...,-27.75,-53.75,-27.8544,-53.7911,0,14.002425,180.0,0,-27.85,-53.75
1,A805,2000-01-02,5.271899,6.899796,0.629632,6.116576,0.836034,-0.506626,100864.770833,100603.4375,...,-27.75,-53.75,-27.8544,-53.7911,0,12.102600,180.0,0,-27.85,-53.75
2,A805,2000-01-03,4.650546,5.955361,1.055964,5.925378,0.746322,-0.582241,100737.343750,100430.6250,...,-27.75,-53.75,-27.8544,-53.7911,0,10.181101,180.0,0,-27.85,-53.75
3,A805,2000-01-04,3.365570,4.957155,0.983495,4.504568,0.213188,-0.383864,100643.036458,100391.2500,...,-27.75,-53.75,-27.8544,-53.7911,0,8.154307,180.0,0,-27.85,-53.75
4,A805,2000-01-05,2.664263,4.092182,0.939953,3.750594,0.017181,-0.051312,100701.078125,100541.1250,...,-27.75,-53.75,-27.8544,-53.7911,0,8.270441,180.0,0,-27.85,-53.75



Formato final do DataFrame: (37988, 35)


In [17]:
print("--- Variáveis Disponíveis (Colunas) ---")
print(df_final.columns.tolist())

print("\n--- Estações Presentes ---")
print(df_final['station'].unique().tolist())

--- Variáveis Disponíveis (Colunas) ---
['station', 'time', 'ws_mean', 'ws_max', 'ws_std', 'ws_p90', 'sin_dir_mean', 'cos_dir_mean', 'msl_mean', 'msl_min', 'msl_range', 't2m_mean', 't2m_max', 't2m_min', 't2m_range', 'rh_mean', 'rh_p10', 'td_dep_mean', 'tp_sum', 'tp_max', 'd_msl', 'tp_lag1', 'tp_lag2', 'tp_roll3', 'rajada_max_final_ms', 'latitude_x', 'longitude_x', 'station_lat', 'station_lon', 'categoria', 'rajada_max_corrigida', 'direcao_vento', 'number', 'latitude_y', 'longitude_y']

--- Estações Presentes ---
['A805', 'A853', 'A854', 'A856']


In [18]:
# Definindo as colunas que serão removidas
# Removendo metadados, IDs, números de membro e coordenadas redundantes
# cols_to_drop = [
#     'station',
#     'number',
#     'latitude_x',
#     'longitude_x',
#     'latitude_y',
#     'longitude_y',
#     'station_lat',
#     'station_lon',
#     'categoria'
# ]

cols_to_drop = [
    'number'
]

# Criando o DataFrame de features (X) removendo as colunas selecionadas
# Mantemos as variáveis meteorológicas e os alvos (rajada_max_final_ms ou rajada_max_corrigida)
df_ml = df_final.drop(columns=cols_to_drop)

print("--- Variáveis Selecionadas para o Modelo ---")
print(df_ml.columns.tolist())

print(f"\nNovo formato do dataset: {df_ml.shape}")
display(df_ml.head())

--- Variáveis Selecionadas para o Modelo ---
['station', 'time', 'ws_mean', 'ws_max', 'ws_std', 'ws_p90', 'sin_dir_mean', 'cos_dir_mean', 'msl_mean', 'msl_min', 'msl_range', 't2m_mean', 't2m_max', 't2m_min', 't2m_range', 'rh_mean', 'rh_p10', 'td_dep_mean', 'tp_sum', 'tp_max', 'd_msl', 'tp_lag1', 'tp_lag2', 'tp_roll3', 'rajada_max_final_ms', 'latitude_x', 'longitude_x', 'station_lat', 'station_lon', 'categoria', 'rajada_max_corrigida', 'direcao_vento', 'latitude_y', 'longitude_y']

Novo formato do dataset: (37988, 34)


,station,time,ws_mean,ws_max,ws_std,ws_p90,sin_dir_mean,cos_dir_mean,msl_mean,msl_min,...,rajada_max_final_ms,latitude_x,longitude_x,station_lat,station_lon,categoria,rajada_max_corrigida,direcao_vento,latitude_y,longitude_y
0,A805,2000-01-01,6.296078,7.452370,0.593588,7.233298,0.739596,-0.659417,100868.335938,100646.3750,...,13.7899,-27.75,-53.75,-27.8544,-53.7911,0,14.002425,180.0,-27.85,-53.75
1,A805,2000-01-02,5.271899,6.899796,0.629632,6.116576,0.836034,-0.506626,100864.770833,100603.4375,...,12.1276,-27.75,-53.75,-27.8544,-53.7911,0,12.102600,180.0,-27.85,-53.75
2,A805,2000-01-03,4.650546,5.955361,1.055964,5.925378,0.746322,-0.582241,100737.343750,100430.6250,...,10.2608,-27.75,-53.75,-27.8544,-53.7911,0,10.181101,180.0,-27.85,-53.75
3,A805,2000-01-04,3.365570,4.957155,0.983495,4.504568,0.213188,-0.383864,100643.036458,100391.2500,...,8.2197,-27.75,-53.75,-27.8544,-53.7911,0,8.154307,180.0,-27.85,-53.75
4,A805,2000-01-05,2.664263,4.092182,0.939953,3.750594,0.017181,-0.051312,100701.078125,100541.1250,...,8.1747,-27.75,-53.75,-27.8544,-53.7911,0,8.270441,180.0,-27.85,-53.75


### 5. Cálculo do PREDEP e Correlações por Estação

In [ ]:
import numpy as np
import scipy.stats as ss
from astropy.stats import bayesian_blocks
import matplotlib.pyplot as plt
import seaborn as sns
import time

def ecdf(data):
    sorted_data = np.sort(data)
    n = len(data)
    def _ecdf(x):
        return np.searchsorted(sorted_data, x, side='right') / n
    return _ecdf

def predep(X, Y, n_boot=5000):
    """PREDEP α_{X|Y}: Y=resposta (particionada), X=preditor (condicionado).
    Retorna quanto Y reduz incerteza sobre X.
    Convenção: X=feature meteorológica, Y=rajada (target).
    """
    try:
        X = np.asarray(X, dtype=float)
        Y = np.asarray(Y, dtype=float)

        BX1 = np.random.choice(X, n_boot)
        BX2 = np.random.choice(X, n_boot)
        diff_x = BX1 - BX2
        if np.std(diff_x) == 0:
            return np.nan
        s_x = float(np.squeeze(ss.gaussian_kde(diff_x).pdf(0)))

        edges_y = bayesian_blocks(Y)
        ecdf_y = ecdf(Y)
        s_x_mid_y = 0.0

        if edges_y.shape[0] > 1:
            for i in range(1, edges_y.shape[0]):
                bg, ed = edges_y[i - 1], edges_y[i]
                X_mid_Y = X[(Y >= bg) & (Y < ed)]
                if X_mid_Y.shape[0] < 2 or len(np.unique(X_mid_Y)) < 2:
                    continue
                BX1_c = np.random.choice(X_mid_Y, n_boot)
                BX2_c = np.random.choice(X_mid_Y, n_boot)
                diff_c = BX1_c - BX2_c
                if np.std(diff_c) == 0:
                    continue
                p_range = ecdf_y(ed) - ecdf_y(bg)
                s_x_mid_y += p_range * float(np.squeeze(ss.gaussian_kde(diff_c).pdf(0)))

            if s_x_mid_y == 0:
                return np.nan
            return float((s_x_mid_y - s_x) / s_x_mid_y)
        return np.nan
    except Exception:
        return np.nan

In [ ]:
features = [
    'ws_mean', 'ws_max', 'ws_std', 'ws_p90', 'sin_dir_mean', 'cos_dir_mean',
    'msl_mean', 'msl_min', 'msl_range', 't2m_mean', 't2m_max', 't2m_min',
    't2m_range', 'rh_mean', 'rh_p10', 'td_dep_mean', 'tp_sum', 'tp_max',
    'd_msl', 'tp_lag1', 'tp_lag2', 'tp_roll3'
]
targets = ['rajada_max_final_ms', 'rajada_max_corrigida']
features = [f for f in features if f in df_ml.columns]

results = []
for station in df_ml['station'].unique():
    df_st = df_ml[df_ml['station'] == station]
    for target in targets:
        for feature in features:
            pair = df_st[[feature, target]].dropna()
            if len(pair) < 50:
                continue
            X = pair[feature].values
            Y = pair[target].values
            t0 = time.time()
            val_predep = predep(X, Y, n_boot=5000)
            corr_p, _ = ss.pearsonr(X, Y)
            corr_s, _ = ss.spearmanr(X, Y)
            results.append({
                'Station': station,
                'Target': target,
                'Feature': feature,
                'PREDEP': round(val_predep, 5) if not np.isnan(val_predep) else np.nan,
                'Pearson': round(corr_p, 4),
                'Spearman': round(corr_s, 4),
                'N_samples': len(X),
                'Calc_Time_s': round(time.time() - t0, 2)
            })

df_predep = pd.DataFrame(results)
display(df_predep)

In [ ]:
for target in targets:
    df_t = df_predep[df_predep['Target'] == target]
    stations = df_t['Station'].unique()
    fig, axes = plt.subplots(len(stations), 1, figsize=(10, 6 * len(stations)))
    if len(stations) == 1:
        axes = [axes]
    for i, station in enumerate(stations):
        ax = axes[i]
        df_st = (
            df_t[df_t['Station'] == station]
            .sort_values('PREDEP', ascending=False)
            .dropna(subset=['PREDEP'])
        )
        if df_st.empty:
            ax.text(0.5, 0.5, f'Sem dados — {station}', ha='center', va='center')
            continue
        sns.barplot(data=df_st, x='PREDEP', y='Feature', ax=ax, palette='viridis')
        ax.set_title(f'PREDEP — {station} | Target: {target}')
        ax.set_xlabel('PREDEP (α)')
        ax.set_ylabel('Feature')
        ax.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()